<img src="http://dask.readthedocs.io/en/latest/_images/dask_horizontal.svg"
     align="right"
     width="30%"
     alt="Dask logo\">

# Futures —— 非阻塞的分布式计算

以并行、立即执行（eager）、非阻塞的方式，把任意函数提交到集群上计算。

`futures` 接口（源自内置的 `concurrent.futures`）为定制场景提供细粒度、实时的执行能力。我们可以用一组输入提交单个函数，也可以用 `submit()` 和 `map()` 在一串输入上求值。调用会立刻返回一个或多个 *future*，状态先是 `"pending"`，随后变成 `"finished"`。本地 Python 会话不会被阻塞。

这是 futures 和 delayed 的重要区别。两者都能做任意任务调度，但 delayed 是惰性的（只构建任务图），而 futures 是立即执行的。使用 futures 时，只要输入就绪且有可用算力，计算就会开始。

**相关文档**

* [Futures 文档](https://docs.dask.org/en/latest/futures.html)
* [Futures 视频](https://www.youtube.com/watch?v=07EiCpdhtDE)
* [Futures 示例](https://examples.dask.org/futures.html)


In [1]:
from dask.distributed import Client

client = Client(n_workers=4)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 15.62 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40573,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 4
Started: Just now,Total memory: 15.62 GiB
Comm: tcp://127.0.0.1:38801,Total threads: 1
Dashboard: http://127.0.0.1:46249/status,Memory: 3.90 GiB
Nanny: tcp://127.0.0.1:39363,


## 一个典型工作流

这和 delayed 那一节看到的工作流相同：偏 for 循环，数据也不一定是 array 或 dataframe。下面是一个“读取—变换—写出”的例子：

```python
def process_file(filename):
    data = read_a_file(filename)
    data = do_a_transformation(data)
    destination = f"results/{filename}"
    write_out_data(data, destination)
    return destination

futures = []
for filename in filenames:
    future = client.submit(process_file, filename)
    futures.append(future)
    
futures
```


## 基础

和 delayed 那一节一样，先写几个玩具函数 `inc` 和 `add`，让它们 `sleep` 一会儿来模拟耗时工作。然后给普通调用计时。


In [2]:
from time import sleep


def inc(x):
    sleep(1)
    return x + 1


def double(x):
    sleep(2)
    return 2 * x


def add(x, y):
    sleep(1)
    return x + y

我们可以在本地运行它们


In [3]:
inc(1)

2

也可以把它们提交给 Dask 远程执行。这会立刻返回一个 future，指向正在进行的计算，最终指向存好的结果。


In [4]:
future = client.submit(inc, 1)  # 立刻返回一个 pending 的 future
future


<Future: pending, key: inc-f5663e0c337416ce03c0edc152f2b488>

等大概一秒再看这个 future，你会发现它已经完成了。


In [5]:
future

<Future: pending, key: inc-f5663e0c337416ce03c0edc152f2b488>

可以用 `.result()` 方法阻塞等待计算，并取出结果。


In [6]:
future.result()

2

#### 等待 future 的其他方式
```python
from dask.distributed import wait, progress
progress(future)
```

会在 *当前* notebook 里显示进度条，而不必跑到仪表盘。这个进度条也是异步的，不会挡住其他代码继续执行。

```python
wait(future)
```
会阻塞，让 notebook 等到 `future` 指向的计算完成。不过要注意：如果 `inc()` 的结果已经待在集群里，现在再执行几乎 **不花时间**，因为 Dask 发现我们要的是它已经知道的那次计算。后面还会再讲。

#### 收集结果的其他方式
```python
client.gather(futures)
```

可以从多个 future 收集结果。


## `client.compute`

一般来说，凡是用 `.compute()` 或 `dask.compute()` 执行的 Dask 操作，都可以改用 `client.compute()` 做异步提交。

这是 delayed 那一节里的例子：


In [7]:
import dask


@dask.delayed
def inc(x):
    sleep(1)
    return x + 1


@dask.delayed
def add(x, y):
    sleep(1)
    return x + y


x = inc(1)
y = inc(2)
z = add(x, y)

到这里我们还只有普通的 `dask.delayed` 输出。把 `z` 传给 `client.compute` 后会得到一个 future，同时 Dask 开始求值任务图。


In [8]:
# 注意它和 z.compute() 的区别
# 注意这个单元格会立刻结束
future = client.compute(z)
future


<Future: pending, key: add-8f490f96-03e3-4cfb-9754-35dfa267a9c6>

In [9]:
future.result()  # 等到结果就绪


5

使用 futures 时，*计算会走到数据所在之处*，而不是把数据搬过来；本地 Python 会话里的 client 甚至不必看到中间值。


## `client.submit`

`client.submit` 接收一个函数和若干参数，把它们推到集群，并返回一个代表待计算结果的 `Future`。函数会被交给某个 worker 进程求值。这看起来很像上面的 `client.compute()`，区别是现在我们把函数和参数直接交给集群。


In [10]:
def inc(x):
    sleep(1)
    return x + 1


future_x = client.submit(inc, 1)
future_y = client.submit(inc, 2)
future_z = client.submit(sum, [future_x, future_y])
future_z

<Future: pending, key: sum-35fa216e61e91664d3d1fb90fc7fde4b>

In [11]:
future_z.result()  # 等到结果就绪


5

传给 `client.submit` 的参数可以是普通 Python 函数和对象、其他 submit 得到的 future，或 `dask.delayed` 对象。


### 它是怎么工作的？

每个 future 都代表集群上已经持有、或正在计算的一个结果。因此我们可以控制中间值的缓存：当一个 future 不再被引用，它的值就会被忘掉。在上面的解答里，每次函数调用都保留了 future。如果我们再提交需要这些结果的新任务，就不必重新计算。

可以用 `client.scatter()` 把本地会话中的数据显式传到集群，但通常更好的做法是：让函数在 worker 内部自己加载数据，这样就不必序列化和传输数据。Dask 里大多数加载函数（例如 `dd.read_csv`）都是这样做的。同样，我们通常也不想 `gather()` 那些内存里过大的结果。


## 示例：偶尔失败的任务

想象一个有时会失败的任务。处理输入数据时你可能遇到这种情况：某个文件格式坏了，或者某次请求超时。


In [12]:
from random import random


def flaky_inc(i):
    if random() < 0.2:
        raise ValueError("You hit the error!")
    return i + 1

反复运行这个函数，它有时会失败。

```python
>>> flaky_inc(2)
---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Input In [65], in <cell line: 1>()
----> 1 flaky_inc(2)

Input In [61], in flaky_inc(i)
      3 def flaky_inc(i):
      4     if random() < 0.5:
----> 5         raise ValueError("You hit the error!")
      6     return i + 1

ValueError: You hit the error!
```


我们可以用 `client.map` 在一组输入上运行这个函数。


In [13]:
futures = client.map(flaky_inc, range(10))

注意：即使有些计算失败了，这个单元格也已经返回。我们可以逐个检查这些 future，找出失败的那些：


In [14]:
for i, future in enumerate(futures):
    print(i, future.status)

0 pending
1 pending
2 pending
3 pending
4 pending
5 pending
6 pending
7 pending
8 pending
9 pending


2026-09-17 06:21:21,523 - distributed.worker - WARNING - Compute Failed
Key:       flaky_inc-c5975878f5000989d064e7873a7e8e0d
Function:  flaky_inc
args:      (9)
kwargs:    {}
Exception: "ValueError('You hit the error!')"

2026-09-17 06:21:21,523 - distributed.worker - WARNING - Compute Failed
Key:       flaky_inc-e830a5c93d53eacd95fba71393826a15
Function:  flaky_inc
args:      (8)
kwargs:    {}
Exception: "ValueError('You hit the error!')"



你可以重跑那些特定的 future，争取让任务成功完成：


In [15]:
futures[5].retry()

In [16]:
for i, future in enumerate(futures):
    print(i, future.status)

0 error
1 error
2 error
3 error
4 finished
5 finished
6 finished
7 finished
8 error
9 error


遇到偶发失败时，更简洁的重试方式是在 `client.compute`、`client.submit` 或 `client.map` 里设置重试次数。

**注意**：这个例子里还需要设置 `pure=False`，告诉 Dask：函数的参数并不能完全决定输出。


In [17]:
futures = client.map(flaky_inc, range(10), retries=5, pure=False)
future_z = client.submit(sum, futures)
future_z.result()

2026-09-17 06:21:21,555 - distributed.worker - WARNING - Compute Failed
Key:       flaky_inc-821e6b49-651d-40c4-89fd-83e2ef659110-0
Function:  flaky_inc
args:      (0)
kwargs:    {}
Exception: "ValueError('You hit the error!')"

2026-09-17 06:21:21,555 - distributed.worker - WARNING - Compute Failed
Key:       flaky_inc-821e6b49-651d-40c4-89fd-83e2ef659110-2
Function:  flaky_inc
args:      (2)
kwargs:    {}
Exception: "ValueError('You hit the error!')"

2026-09-17 06:21:21,556 - distributed.worker - WARNING - Compute Failed
Key:       flaky_inc-821e6b49-651d-40c4-89fd-83e2ef659110-5
Function:  flaky_inc
args:      (5)
kwargs:    {}
Exception: "ValueError('You hit the error!')"

2026-09-17 06:21:21,557 - distributed.worker - WARNING - Compute Failed
Key:       flaky_inc-821e6b49-651d-40c4-89fd-83e2ef659110-4
Function:  flaky_inc
args:      (4)
kwargs:    {}
Exception: "ValueError('You hit the error!')"

2026-09-17 06:21:21,557 - distributed.worker - WARNING - Compute Failed
Key:       f

55

你会看到很多警告，但计算最终应当能成功。


## 为什么使用 Futures？

futures API 提供一种“提交工作”的风格，很容易模仿 map/reduce。如果你熟悉那套范式，futures 可能是进入 Dask 最简单的入口。

futures 的另一个大好处是：由 future 表示的中间结果可以直接传给新任务，而不必把数据从集群拉回本地。新操作甚至可以建立在那些还没开始的旧任务的输出上。
